<h1><center>Titanic Survival Prediction</center></h1>
<hr>


<h1>Objective</h1>
<p>This notebook walks through an end-to-end Titanic survival prediction workflow. The goal is to explore the data, engineer useful features, and compare machine learning models using cross-validation.</p>
<hr>


**Importing Libraries + Data**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
training = pd.read_csv('titanic-train.csv')

%matplotlib inline
training.columns


<h1>Project Planning</h1>
<hr>

1. Understand the structure of the data with `.info()` and `.describe()`
2. Explore numeric distributions with histograms and boxplots
3. Review category counts for key features
4. Check for missing values
5. Study correlations between numeric variables
6. Explore survival patterns by class, embarkation point, age, and fare
7. Engineer features from cabin, ticket, and passenger title
8. Build a preprocessing pipeline that avoids leakage during evaluation
9. Compare baseline models with cross-validation
10. Tune the strongest models with a compact parameter search


<h1>Exploratory Data Analysis</h1>
<hr>

In [ ]:
training.sample(5)
training.info()

In [ ]:
training.describe()
training.describe().columns

In [ ]:
df_num = training[['Age','SibSp','Parch','Fare']]
df_cat = training[['Survived','Pclass','Sex','Ticket','Cabin','Embarked']]

In [ ]:
print(df_num.corr())
sns.heatmap(df_num.corr(), cmap="YlGnBu")

In [ ]:
#compare survival rates across Age, SibSp, Parch, Fare
pd.pivot_table(training, index = 'Survived', values = ['Age','SibSp','Parch','Fare'])

In [ ]:
for i in df_cat.columns:
    counts = df_cat[i].value_counts()

    sns.barplot(x=counts.index, y=counts.values)
    plt.title(i)
    plt.show()

In [ ]:
# Comparing survival and each of these categorical variables 
print(pd.pivot_table(training, index = 'Survived', columns = 'Pclass', values = 'Ticket' ,aggfunc ='count'))
print()
print(pd.pivot_table(training, index = 'Survived', columns = 'Sex', values = 'Ticket' ,aggfunc ='count'))
print()
print(pd.pivot_table(training, index = 'Survived', columns = 'Embarked', values = 'Ticket' ,aggfunc ='count'))

**Feature Engineering**

In [ ]:
df_cat.Cabin
training['cabin_multiple'] = training.Cabin.apply(lambda x: 0 if pd.isna(x) else len(x.split(' ')))
# after looking at this, we may want to look at cabin by letter or by number. Let's create some categories for this 
# letters 
# multiple letters 
training['cabin_multiple'].value_counts()

In [ ]:
pd.pivot_table(training, index = 'Survived', columns = 'cabin_multiple', values = 'Ticket' ,aggfunc ='count')

In [ ]:
#creates categories based on the cabin letter (n stands for null)
#in this case we will treat null values like it's own category

training['cabin_adv'] = training.Cabin.apply(lambda x: str(x)[0])

In [ ]:
# comparing survival rate by cabin
print(training.cabin_adv.value_counts())
pd.pivot_table(training,index='Survived',columns='cabin_adv', values = 'Name', aggfunc='count')


In [ ]:
#understand ticket values better 
#numeric vs non numeric 
training['numeric_ticket'] = training.Ticket.apply(lambda x: 1 if x.isnumeric() else 0)
training['ticket_letters'] = training.Ticket.apply(lambda x: ''.join(x.split(' ')[:-1]).replace('.','').replace('/','').lower() if len(x.split(' ')[:-1]) >0 else 0)

In [ ]:
training['numeric_ticket'].value_counts()

In [ ]:
#difference in numeric vs non-numeric tickets in survival rate 
pd.pivot_table(training,index='Survived',columns='numeric_ticket', values = 'Ticket', aggfunc='count')

In [ ]:
#feature engineering on person's title 
training.Name.head(50)
training['name_title'] = training.Name.apply(lambda x: x.split(',')[1].split('.')[0].strip())
#mr., ms., master. etc

In [ ]:
training['name_title'].value_counts()


<h1>Train-Only Preprocessing and Evaluation</h1>
<hr>

From this point onward, all evaluation statistics are computed with preprocessing fit only on training folds inside cross-validation. This keeps the workflow focused on honest model comparison and avoids leakage from combined preprocessing.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC


In [ ]:
def extract_title(name):
    if pd.isna(name):
        return "Unknown"
    parts = str(name).split(",")
    if len(parts) < 2 or "." not in parts[1]:
        return "Unknown"
    return parts[1].split(".")[0].strip()


def is_numeric_ticket(ticket):
    cleaned = str(ticket).replace(".", "").replace("/", "").replace(" ", "")
    return int(cleaned.isnumeric())


def add_features(df):
    df = df.copy()
    df["cabin_multiple"] = df["Cabin"].fillna("").apply(lambda x: 0 if x == "" else len(str(x).split()))
    df["cabin_adv"] = df["Cabin"].fillna("Missing").astype(str).str[0]
    df["numeric_ticket"] = df["Ticket"].apply(is_numeric_ticket)
    df["name_title"] = df["Name"].apply(extract_title)
    df["fare_log"] = np.log1p(df["Fare"])
    df["Pclass"] = df["Pclass"].astype(str)
    return df


train_model_df = add_features(training)

feature_cols = [
    "Pclass", "Sex", "Age", "SibSp", "Parch", "Embarked",
    "cabin_multiple", "cabin_adv", "numeric_ticket", "name_title", "fare_log"
]

X = train_model_df[feature_cols]
y = train_model_df["Survived"].copy()

X.head()


In [ ]:
numeric_features = ["Age", "SibSp", "Parch", "cabin_multiple", "numeric_ticket", "fare_log"]
categorical_features = ["Pclass", "Sex", "Embarked", "cabin_adv", "name_title"]

preprocess = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]
            ),
            numeric_features
        ),
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore"))
                ]
            ),
            categorical_features
        )
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
pd.DataFrame(
    {
        "missing_values": X.isna().sum(),
        "dtype": X.dtypes.astype(str)
    }
).sort_values("missing_values", ascending=False)


<h1>Model Building and Evaluation</h1>
<hr>

We compare four models using the same 5-fold stratified cross-validation setup. Each pipeline fits imputation, encoding, and scaling inside the fold, which avoids train/test leakage during evaluation.


In [ ]:
baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="liblinear", random_state=42),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVC": SVC(random_state=42)
}

baseline_rows = []

for model_name, estimator in baseline_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", estimator)
        ]
    )

    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        return_train_score=False
    )

    print(model_name, "fold accuracies:", np.round(scores["test_score"], 3))
    print(model_name, "mean accuracy:", round(scores["test_score"].mean(), 4))
    print()

    baseline_rows.append(
        {
            "model": model_name,
            "baseline_mean_accuracy": scores["test_score"].mean(),
            "baseline_std": scores["test_score"].std()
        }
    )

baseline_results = (
    pd.DataFrame(baseline_rows)
    .sort_values("baseline_mean_accuracy", ascending=False)
    .reset_index(drop=True)
)

baseline_results.style.format(
    {
        "baseline_mean_accuracy": "{:.3f}",
        "baseline_std": "{:.3f}"
    }
)


<h1>Compact Hyperparameter Tuning</h1>
<hr>

To keep the notebook executable, the search grids stay intentionally small. The goal is to improve the main models without turning the notebook into a long-running experimentation script.


In [ ]:
param_grids = {
    "Logistic Regression": {
        "model__C": [0.1, 1.0, 3.0, 10.0],
        "model__penalty": ["l1", "l2"],
        "model__solver": ["liblinear"]
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__p": [1, 2]
    },
    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 4, 8],
        "model__min_samples_split": [2, 5],
        "model__min_samples_leaf": [1, 2]
    },
    "SVC": {
        "model__C": [0.5, 1.0, 2.0, 5.0],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    }
}

tuned_rows = []

for model_name, estimator in baseline_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", estimator)
        ]
    )

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[model_name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1
    )
    search.fit(X, y)

    tuned_scores = cross_validate(
        search.best_estimator_,
        X,
        y,
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        return_train_score=False
    )

    print(model_name, "best search score:", round(search.best_score_, 4))
    print(model_name, "re-evaluated tuned accuracy:", round(tuned_scores["test_score"].mean(), 4))
    print(model_name, "best params:", search.best_params_)
    print()

    tuned_rows.append(
        {
            "model": model_name,
            "tuned_mean_accuracy": tuned_scores["test_score"].mean(),
            "tuned_std": tuned_scores["test_score"].std(),
            "best_params": str(search.best_params_)
        }
    )

tuned_results = pd.DataFrame(tuned_rows)
tuned_results


In [ ]:
comparison = (
    baseline_results.merge(tuned_results, on="model", how="left")
    .assign(improvement=lambda df: df["tuned_mean_accuracy"] - df["baseline_mean_accuracy"])
    .sort_values("tuned_mean_accuracy", ascending=False)
    .reset_index(drop=True)
)

comparison.style.format(
    {
        "baseline_mean_accuracy": "{:.3f}",
        "baseline_std": "{:.3f}",
        "tuned_mean_accuracy": "{:.3f}",
        "tuned_std": "{:.3f}",
        "improvement": "{:+.3f}"
    }
)


<h1>Final Takeaway</h1>
<hr>

This cleaned workflow keeps evaluation honest by fitting preprocessing only inside each cross-validation fold. The final comparison table highlights which models benefited most from tuning, while keeping the project focused on reproducible statistics instead of model export or submission generation.
